# Fine-Grained Tool Calling (Lesson 29)

Follow-along notebook for *Fine grained tool calling* in Building with the Claude API.

**What you'll see:**
1. Basic tool streaming: `input_json` events with `partial_json` and `snapshot`
2. Default mode: the API buffers and validates each top-level key, so tool input arrives in **bursts**
3. Fine-grained mode: chunks arrive as Claude generates them, **unvalidated**
4. Handling invalid JSON when validation is off

**Run order:** Run All after a kernel restart.

> **API note:** the lesson's helper takes `fine_grained=True`. Under the hood, the current API turns this on per tool by adding `"eager_input_streaming": True` to the tool definition, which is what `stream_chat` below does. If your SDK is old, run `pip install -U anthropic` in your venv.

## 1. Setup

In [1]:
import json
import time

from anthropic import Anthropic
from anthropic.types import ToolParam
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()
MODEL = "claude-haiku-4-5"

## 2. The tool: `save_article`
A nested input (`abstract` plus a `meta` object) makes the buffering behavior easy to see.

In [2]:
def save_article(abstract, meta):
    """Stand-in for saving to a database: just prints what it received."""
    print("\n---- save_article called ----")
    print(f"Abstract ({len(abstract.split())} words): {abstract[:150]}...")
    print(f"Meta: {meta}")
    print("-----------------------------")
    return "Article saved"


save_article_schema = ToolParam({
    "name": "save_article",
    "description": "Saves a research article abstract along with metadata about it.",
    "input_schema": {
        "type": "object",
        "properties": {
            "abstract": {
                "type": "string",
                "description": "The full text of the abstract.",
            },
            "meta": {
                "type": "object",
                "properties": {
                    "word_count": {
                        "type": "integer",
                        "description": "Number of words in the abstract.",
                    },
                    "review": {
                        "type": "string",
                        "description": "A short critical review of the abstract.",
                    },
                },
                "required": ["word_count", "review"],
            },
        },
        "required": ["abstract", "meta"],
    },
})

PROMPT = (
    "Write a roughly 300-word abstract for a fictional research paper about a new "
    "quantum networking protocol called QuanNet. Then save it with the save_article tool, "
    "including its word count and a short review."
)

## 3. Basic tool streaming (the lesson's first snippet)

Uses the SDK's `client.messages.stream` helper. `tool_choice` forces Claude to call the tool so there's tool input to watch.

- `partial_json`: just the new chunk
- `snapshot`: everything received so far, already parsed by the SDK into a Python object

Watch how the output pauses, then dumps a whole `abstract` at once: that's server-side buffering.

In [5]:
messages = [{"role": "user", "content": PROMPT}]

with client.messages.stream(
    model=MODEL,
    max_tokens=2000,
    messages=messages,
    tools=[save_article_schema],
    tool_choice={"type": "tool", "name": "save_article"},
) as stream:
    for chunk in stream:
        if chunk.type == "input_json":
            print(chunk.partial_json, end="", flush=True)

    final = stream.get_final_message()

print("\n\nLast snapshot keys:", list(final.content[-1].input.keys()))

{"abstract": "QuanNet: A Novel Quantum Networking Protocol for Secure Distributed Computing\n\nAbstract: The development of practical quantum networking infrastructure remains a critical challenge in realizing large-scale quantum information systems. This paper introduces QuanNet, a revolutionary quantum networking protocol designed to enable secure, efficient communication between multiple quantum processing nodes across geographically distributed networks. QuanNet addresses fundamental challenges in quantum state transmission, entanglement distribution, and quantum error correction through a novel multi-layered protocol architecture that seamlessly integrates classical control channels with quantum data channels.\n\nOur protocol employs a hierarchical approach to quantum repeater stations, utilizing a combination of matter qubits and photonic qubits to maximize the fidelity of long-distance entanglement distribution. The key innovation of QuanNet lies in its adaptive routing algorith

## 4. Streaming chat helper with timing

`stream_chat` uses the raw event stream (`stream=True`) and builds the reply itself, so it can:
- print a **timestamp** on every tool-input chunk, which makes bursts vs. steady streaming visible
- turn on fine-grained mode with `fine_grained=True`
- survive **invalid JSON** instead of crashing (the SDK helper in section 3 parses as it goes, so it can raise on bad JSON)

It returns `(content_blocks, stop_reason)`.

In [6]:
def stream_chat(messages, tools=None, fine_grained=False, system=None, show_json=True):
    params = {"model": MODEL, "max_tokens": 2000, "messages": messages, "stream": True}
    if system:
        params["system"] = system
    if tools:
        # Fine-grained mode is enabled per tool with eager_input_streaming
        params["tools"] = (
            [dict(t, eager_input_streaming=True) for t in tools] if fine_grained else list(tools)
        )

    blocks = {}     # index -> content block being built
    raw_json = {}   # index -> accumulated tool-input JSON string
    stop_reason = None
    start = time.perf_counter()

    for event in client.messages.create(**params):
        elapsed = time.perf_counter() - start

        if event.type == "content_block_start":
            cb = event.content_block
            if cb.type == "text":
                blocks[event.index] = {"type": "text", "text": ""}
            elif cb.type == "tool_use":
                blocks[event.index] = {"type": "tool_use", "id": cb.id, "name": cb.name, "input": {}}
                raw_json[event.index] = ""
                print(f"\n[{elapsed:6.2f}s] tool_use started: {cb.name}")

        elif event.type == "content_block_delta" and event.index in blocks:
            delta = event.delta
            if delta.type == "text_delta":
                blocks[event.index]["text"] += delta.text
                print(delta.text, end="", flush=True)
            elif delta.type == "input_json_delta":
                raw_json[event.index] += delta.partial_json
                if show_json and delta.partial_json:
                    print(f"[{elapsed:6.2f}s] {delta.partial_json!r}")

        elif event.type == "content_block_stop" and event.index in raw_json:
            block = blocks[event.index]
            raw = raw_json[event.index] or "{}"
            try:
                block["input"] = json.loads(raw)
            except json.JSONDecodeError as e:
                # Only possible with fine-grained mode: the API did not validate this
                block["_invalid_json"] = raw
                print(f"\n[{elapsed:6.2f}s] Invalid JSON from Claude: {e}")

        elif event.type == "message_delta":
            stop_reason = event.delta.stop_reason

    print()
    content = [blocks[i] for i in sorted(blocks)]
    return content, stop_reason

## 5. Tool router and conversation loop
Same pattern as the reminder notebook, with streaming and an invalid-JSON path that sends an error back to Claude so it can retry.

In [7]:
def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)
    raise ValueError(f"Unknown tool: {tool_name}")


def run_tools(content):
    results = []
    for block in content:
        if block["type"] != "tool_use":
            continue
        if "_invalid_json" in block:
            results.append({
                "type": "tool_result",
                "tool_use_id": block["id"],
                "content": "Your tool input was not valid JSON: "
                           f"{block['_invalid_json'][:500]}\nPlease call the tool again with valid JSON.",
                "is_error": True,
            })
            continue
        try:
            output = run_tool(block["name"], block["input"])
            results.append({"type": "tool_result", "tool_use_id": block["id"],
                            "content": json.dumps(output), "is_error": False})
        except Exception as e:
            results.append({"type": "tool_result", "tool_use_id": block["id"],
                            "content": f"Error: {e}", "is_error": True})
    return results


def clean_for_api(content):
    """Drop our private keys and any empty text blocks before sending history back."""
    cleaned = []
    for block in content:
        if block["type"] == "text" and not block["text"].strip():
            continue
        cleaned.append({k: v for k, v in block.items() if not k.startswith("_")})
    return cleaned


def run_conversation(messages, tools, fine_grained=False, max_turns=10):
    for _ in range(max_turns):
        content, stop_reason = stream_chat(messages, tools=tools, fine_grained=fine_grained)
        messages.append({"role": "assistant", "content": clean_for_api(content)})

        if stop_reason != "tool_use":
            break
        messages.append({"role": "user", "content": run_tools(content)})
    return messages

## 6. Default mode: buffered and validated

Look at the timestamps on the tool-input chunks. Expect a gap while `abstract` is generated, then many chunks with nearly the same timestamp: the API held them until the whole top-level value was complete and valid.

In [8]:
messages = [{"role": "user", "content": PROMPT}]
run_conversation(messages, tools=[save_article_schema], fine_grained=False);


[  0.79s] tool_use started: save_article
[  4.96s] '{"abstr'
[  4.96s] 'act": "'
[  4.96s] 'Quantum'
[  4.96s] ' networ'
[  4.96s] 'king re'
[  4.96s] 'present'
[  4.96s] 's the n'
[  4.96s] 'ext fro'
[  4.96s] 'ntier i'
[  4.96s] 'n secur'
[  4.96s] 'e and e'
[  4.96s] 'fficien'
[  4.96s] 't commu'
[  4.96s] 'nicatio'
[  4.96s] 'n syste'
[  4.96s] 'ms. In '
[  4.96s] 'this pa'
[  4.96s] 'per, we'
[  4.96s] ' introd'
[  4.96s] 'uce Qua'
[  4.96s] 'nNet, a'
[  4.96s] ' novel '
[  4.96s] 'quantum'
[  4.96s] ' networ'
[  4.96s] 'king pr'
[  4.96s] 'otocol '
[  4.96s] 'designe'
[  4.96s] 'd to fa'
[  4.96s] 'cilitat'
[  4.96s] 'e scala'
[  4.96s] 'ble, lo'
[  4.96s] 'ng-dist'
[  4.96s] 'ance en'
[  4.96s] 'tanglem'
[  4.96s] 'ent dis'
[  4.96s] 'tributi'
[  4.96s] 'on acro'
[  4.96s] 'ss hete'
[  4.96s] 'rogeneo'
[  4.96s] 'us quan'
[  4.96s] 'tum net'
[  4.96s] 'work ar'
[  4.96s] 'chitect'
[  4.96s] 'ures. Q'
[  4.96s] 'uanNet '
[  4.96s] 'address'
[  4.96s] 'es crit'
[  4.96s] 'ical ch

## 7. Fine-grained mode: unbuffered, unvalidated

Same request with `fine_grained=True`. Chunks should arrive steadily from the moment Claude starts the tool input, and `word_count` can show up without waiting for the whole `meta` object.

In [9]:
messages = [{"role": "user", "content": PROMPT}]
run_conversation(messages, tools=[save_article_schema], fine_grained=True);

I'll create a fictional abstract about QuanNet and save it for you.
[  0.48s] tool_use started: save_article
[  0.53s] '{"abstract": "QuanNet: A'
[  0.55s] ' Scal'
[  0.58s] 'able Quantum Networking Protocol'
[  0.60s] ' for Distributed Ent'
[  0.65s] 'anglement Distribution'
[  0.65s] '\\n\\nAbstract:'
[  0.67s] ' The'
[  0.70s] ' re'
[  0.72s] 'alization of large'
[  0.75s] '-scale quantum networks requires robust'
[  0.77s] ' protocols'
[  0.80s] ' for establishing'
[  0.82s] ' and'
[  0.84s] ' maintaining quantum'
[  0.87s] ' ent'
[  0.89s] 'ang'
[  0.92s] 'lement across'
[  0.96s] ' multiple'
[  0.96s] ' nodes while'
[  0.99s] ' minim'
[  1.01s] 'izing de'
[  1.04s] 'coherence and communication'
[  1.07s] ' overhead. This paper presents Qu'
[  1.09s] 'anNet, a novel quantum'
[  1.12s] ' networking protocol designed'
[  1.14s] ' to enable efficient ent'
[  1.17s] 'anglement distribution in heter'
[  1.19s] 'ogeneous quantum'
[  1.21s] ' network top'
[  1.24s] 'ologies. Qu'
[  1.27s

## 8. Handling invalid JSON

Claude rarely produces invalid JSON, so this cell simulates it with the lesson's example (`undefined` instead of a number) to show the handling path without depending on luck.

In [10]:
bad_snapshot = '{"abstract": "QuanNet is...", "meta": {"word_count": undefined, "review": "Solid"}}'

try:
    parsed_args = json.loads(bad_snapshot)
except json.JSONDecodeError as e:
    print(f"Received invalid JSON, continuing... ({e})")

# What run_tools would send back to Claude for a block like this:
fake_block = {"type": "tool_use", "id": "toolu_demo", "name": "save_article",
              "input": {}, "_invalid_json": bad_snapshot}
print(json.dumps(run_tools([fake_block]), indent=2))

Received invalid JSON, continuing... (Expecting value: line 1 column 54 (char 53))
[
  {
    "type": "tool_result",
    "tool_use_id": "toolu_demo",
    "content": "Your tool input was not valid JSON: {\"abstract\": \"QuanNet is...\", \"meta\": {\"word_count\": undefined, \"review\": \"Solid\"}}\nPlease call the tool again with valid JSON.",
    "is_error": true
  }
]


## Takeaways
- Default streaming validates each **top-level** key before sending it, which causes delay-then-burst behavior.
- Fine-grained mode removes that validation: faster, smoother chunks, but your code owns JSON error handling.
- For most apps the default is fine; use fine-grained when users see tool input being built or you need to act on partial results early.